In [1]:
from google.colab import drive
drive.mount('/gdrive')
drive_root = '/gdrive/My Drive/strykthon'

COLAB_DIR = '/content/data'
GT_DIR = COLAB_DIR + '/data/gtFine/'
IMG_DIR = COLAB_DIR + '/data/leftImg8bit/'

Drive already mounted at /gdrive; to attempt to forcibly remount, call drive.mount("/gdrive", force_remount=True).


In [2]:
import torch

if not torch.cuda.is_available():
    print("No GPU device available. Please ensure that a GPU is properly connected and configured.")
    exit()

No GPU device available. Please ensure that a GPU is properly connected and configured.


In [5]:
!pip install cityscapesscripts

In [4]:
from cityscapesscripts.download import downloader

In [6]:
# registration on https://www.cityscapes-dataset.com/
session = downloader.login()
downloader.get_available_packages(session=session)

# data for semantic segmentation task
print('Downloading gtFine and leftImg8bit packages ...\n')
package_list =['gtFine_trainvaltest.zip', 'leftImg8bit_trainvaltest.zip']
downloader.download_packages(session=session, package_names=package_list, destination_path=COLAB_DIR)

In [ ]:
from torchvision.datasets import Cityscapes

In [ ]:
from matplotlib import pyplot as plt
from PIL import Image
import numpy as np
import torch
import torch.nn as nn

In [ ]:
dataset=Cityscapes('./data/',split='train',mode='fine',target_type='semantic')

In [ ]:
dataset[0][0].size

In [ ]:
fig,ax=plt.subplots(ncols=2,figsize=(12,8))
ax[0].imshow(dataset[0][0])
ax[1].imshow(dataset[0][1],cmap='gray')

In [ ]:
ignore_index=255
void_classes = [0, 1, 2, 3, 4, 5, 6, 9, 10, 14, 15, 16, 18, 29, 30, -1]
valid_classes = [ignore_index,7, 8, 11, 12, 13, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 31, 32, 33]
class_names = ['unlabelled', 'road', 'sidewalk', 'building', 'wall', 'fence', 'pole', 'traffic_light', \
               'traffic_sign', 'vegetation', 'terrain', 'sky', 'person', 'rider', 'car', 'truck', 'bus', \
               'train', 'motorcycle', 'bicycle']
#why i choose 20 classes
#https://stackoverflow.com/a/64242989

class_map = dict(zip(valid_classes, range(len(valid_classes))))
n_classes=len(valid_classes)
class_map

In [ ]:
colors = [   [  0,   0,   0],
        [128, 64, 128],
        [244, 35, 232],
        [70, 70, 70],
        [102, 102, 156],
        [190, 153, 153],
        [153, 153, 153],
        [250, 170, 30],
        [220, 220, 0],
        [107, 142, 35],
        [152, 251, 152],
        [0, 130, 180],
        [220, 20, 60],
        [255, 0, 0],
        [0, 0, 142],
        [0, 0, 70],
        [0, 60, 100],
        [0, 80, 100],
        [0, 0, 230],
        [119, 11, 32],
    ]

label_colours = dict(zip(range(n_classes), colors))

In [ ]:
def encode_segmap(mask):
    #remove unwanted classes and recitify the labels of wanted classes
    for _voidc in void_classes:
        mask[mask == _voidc] = ignore_index
    for _validc in valid_classes:
        mask[mask == _validc] = class_map[_validc]
    return mask

In [ ]:
def decode_segmap(temp):
    #convert gray scale to color
    temp=temp.numpy()
    r = temp.copy()
    g = temp.copy()
    b = temp.copy()
    for l in range(0, n_classes):
        r[temp == l] = label_colours[l][0]
        g[temp == l] = label_colours[l][1]
        b[temp == l] = label_colours[l][2]

    rgb = np.zeros((temp.shape[0], temp.shape[1], 3))
    rgb[:, :, 0] = r / 255.0
    rgb[:, :, 1] = g / 255.0
    rgb[:, :, 2] = b / 255.0
    return rgb

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
transform=A.Compose(
[
    A.Resize(512, 512),
    A.HorizontalFlip(),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
]
)

In [ ]:
from typing import Any, Callable, Dict, List, Optional, Union, Tuple
from torchvision.datasets import Cityscapes

class MyClass(Cityscapes):
    def __getitem__(self, index: int) -> Tuple[Any, Any]:
        image = Image.open(self.images[index]).convert('RGB')

        targets: Any = []
        for i, t in enumerate(self.target_type):
            if t == 'polygon':
                target = self._load_json(self.targets[index][i])
            else:
                target = Image.open(self.targets[index][i])
            targets.append(target)
        target = tuple(targets) if len(targets) > 1 else targets[0]

        if self.transforms is not None:
            transformed=transform(image=np.array(image), mask=np.array(target))
        return transformed['image'],transformed['mask']

In [ ]:
dataset=MyClass('./data/', split='val', mode='fine',
                     target_type='semantic',transforms=transform)
img,seg= dataset[20]
print(img.shape,seg.shape)

In [ ]:
fig,ax=plt.subplots(ncols=2,nrows=1,figsize=(16,8))
ax[0].imshow(img.permute(1, 2, 0))
ax[1].imshow(seg,cmap='gray')

In [ ]:
#class labels before label correction
print(torch.unique(seg))
print(len(torch.unique(seg)))

In [ ]:
#class labels after label correction
res=encode_segmap(seg.clone())
print(res.shape)
print(torch.unique(res))
print(len(torch.unique(res)))

In [ ]:
res1=decode_segmap(res.clone())

In [ ]:
fig,ax=plt.subplots(ncols=2,figsize=(12,10))
ax[0].imshow(res,cmap='gray')
ax[1].imshow(res1)

In [ ]:
!pip install segmentation_models_pytorch
!pip install pytorch-lightning
!pip install torchmetrics

In [ ]:
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from pytorch_lightning.callbacks import EarlyStopping,ModelCheckpoint,LearningRateMonitor
import segmentation_models_pytorch as smp

In [ ]:
import multiprocessing
import torch
import segmentation_models_pytorch as smp
from torchmetrics import JaccardIndex
from pytorch_lightning import LightningModule, Trainer
from torch.utils.data import DataLoader
from torch.optim import AdamW

class OurModel(LightningModule):
    def __init__(self):
        super(OurModel, self).__init__()

        # Model architecture: UNet with a ResNet-34 encoder, pretrained on ImageNet
        self.layer = smp.Unet(
            encoder_name="resnet34",
            encoder_weights="imagenet",
            in_channels=3,
            classes=n_classes,
        )

        # Parameters
        self.lr = 1e-3
        self.batch_size = 16
        self.num_workers = multiprocessing.cpu_count() // 4  # Adjust the number of workers

        # Loss and metric
        self.criterion = smp.losses.DiceLoss(mode='multiclass')
        self.metrics = JaccardIndex(num_classes=n_classes, task='multiclass')

        # Define data classes for training and validation
        self.train_class = MyClass(
            './data/', split='train', mode='fine',
            target_type='semantic', transforms=transform
        )
        self.val_class = MyClass(
            './data/', split='val', mode='fine',
            target_type='semantic', transforms=transform
        )

    def process(self, image, segment):
        # Forward pass
        out = self(image)
        segment = encode_segmap(segment)

        # Calculate loss and IoU metric
        loss = self.criterion(out, segment.long())
        iou = self.metrics(out, segment)
        return loss, iou

    def forward(self, x):
        # Forward method
        return self.layer(x)

    def configure_optimizers(self):
        # Optimizer configuration
        opt = AdamW(self.parameters(), lr=self.lr)
        return opt

    def train_dataloader(self):
        # Training data loader
        return DataLoader(
            self.train_class,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=True,
        )

    def training_step(self, batch, batch_idx):
        # Training step
        image, segment = batch
        loss, iou = self.process(image, segment)
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('train_iou', iou, on_step=False, on_epoch=True, prog_bar=False)
        return loss

    def val_dataloader(self):
        # Validation data loader
        return DataLoader(
            self.val_class,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
        )

    def validation_step(self, batch, batch_idx):
        # Validation step
        image, segment = batch
        loss, iou = self.process(image, segment)
        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=False)
        self.log('val_iou', iou, on_step=False, on_epoch=True, prog_bar=False)
        return loss


In [ ]:
!pip install -U pytorch-lightning
import torchmetrics
model = OurModel()
checkpoint_callback = ModelCheckpoint(monitor='val_loss',dirpath='checkpoints',
                                      filename='file',save_last=True)

In [ ]:
from pytorch_lightning import Trainer
trainer = Trainer(max_epochs=10, accelerator="gpu", devices=1)

trainer.fit(model)